# Focal loss experiment (gamma=2)

First focal-loss run on the CE stack before hyperparameter sweeps.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import importlib.util
import os
import sys
from pathlib import Path

def _load_project():
    my_drive = Path("/content/drive/MyDrive")
    init_candidates = [
        p for p in my_drive.rglob("colab_init.py")
        if (p.parent / "birdclef").is_dir()
    ]
    if init_candidates:
        init_path = min(init_candidates, key=lambda p: len(p.parts))
        spec = importlib.util.spec_from_file_location("colab_init", init_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return
    roots = [
        nb.parent.resolve()
        for nb in my_drive.rglob("02_download_and_extract_embeddings.ipynb")
        if (nb.parent / "birdclef").is_dir()
    ]
    if not roots:
        raise FileNotFoundError(
            "Unzip the full repository on Google Drive, then open a notebook from that folder."
        )
    root = min(roots, key=lambda p: len(p.parts))
    sys.path.insert(0, str(root))
    os.chdir(root)

_load_project()

!pip install -q onnxscript onnxruntime-gpu soundfile tqdm scikit-learn matplotlib seaborn kaggle

from birdclef.colab import colab_prepare_training

colab_prepare_training(stage_tta=False)


Mounted at /content/drive
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.3 MB/s eta 0:00:00
Copied train.csv to /content/train.csv
Copied sample_submission.csv to /content/sample_submission.csv
Unzipped baseline embeddings to /content/embeddings_v2
Project root: /content/drive/MyDrive/BirdCLEF_Project/repro


In [2]:
import os
import json
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import (
    BirdDataset,
    BirdMoE,
    FocalLoss,
    competition_macro_roc_auc,
    plot_and_save_learning_curves,
    prepare_baseline_data,
    prepare_tta_data,
    seed_everything,
)
from birdclef.paths import MODELS_DIR

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda


In [3]:
df, NUM_CLASSES, _ = prepare_baseline_data()
model = BirdMoE(input_dim=1536, num_classes=NUM_CLASSES).to(device)


In [4]:
SAVE_DIR = MODELS_DIR / "focal_gamma2_10ep"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
criterion = FocalLoss(gamma=2.0)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 10
focal_fold_scores = []
best_auc = 0.0

for TRAIN_FOLD in range(5):
    train_df = df[df["fold"] != TRAIN_FOLD].reset_index(drop=True)
    valid_df = df[df["fold"] == TRAIN_FOLD].reset_index(drop=True)
    train_ds = BirdDataset(train_df, is_tta=False)
    valid_ds = BirdDataset(valid_df, is_tta=False)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False, num_workers=0)
    print(f"TRAINING FOLD {TRAIN_FOLD}")
    print(f"Training on {len(train_df)} samples, validating on {len(valid_df)} samples")
    fold_best = 0.0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                all_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        avg_val_loss = val_loss / len(valid_loader)
        val_preds = np.concatenate(all_preds)
        val_labels = np.concatenate(all_labels)
        val_labels_onehot = np.eye(NUM_CLASSES)[val_labels]
        current_auc = competition_macro_roc_auc(val_labels_onehot, val_preds)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val AUC: {current_auc:.4f}"
        )
        if current_auc > fold_best:
            fold_best = current_auc
            save_path = SAVE_DIR / f"best_moe_fold{TRAIN_FOLD}.pth"
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path}")

    focal_fold_scores.append(fold_best)
    best_auc = max(best_auc, fold_best)

print(f"CV score: {np.mean(focal_fold_scores):.4f} (+/- {np.std(focal_fold_scores):.4f})")


100%|██████████| 7110/7110 [00:01<00:00, 3664.66it/s]


TRAINING FOLD 0
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.0262 | Val Loss: 0.6922 | Val AUC: 0.9901
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold0.pth
Epoch 02/10 | Train Loss: 0.3736 | Val Loss: 0.6868 | Val AUC: 0.9897
Epoch 03/10 | Train Loss: 0.2008 | Val Loss: 0.7193 | Val AUC: 0.9898
Epoch 04/10 | Train Loss: 0.1424 | Val Loss: 0.7562 | Val AUC: 0.9876
Epoch 05/10 | Train Loss: 0.1144 | Val Loss: 0.7657 | Val AUC: 0.9881
Epoch 06/10 | Train Loss: 0.1144 | Val Loss: 0.7708 | Val AUC: 0.9898
Epoch 07/10 | Train Loss: 0.1158 | Val Loss: 0.7722 | Val AUC: 0.9881
Epoch 08/10 | Train Loss: 0.1061 | Val Loss: 0.7668 | Val AUC: 0.9895
Epoch 09/10 | Train Loss: 0.0978 | Val Loss: 0.7739 | Val AUC: 0.9882
Epoch 10/10 | Train Loss: 0.0855 | Val Loss: 0.7773 | Val AUC: 0.9882


100%|██████████| 7110/7110 [00:02<00:00, 3186.56it/s]


TRAINING FOLD 1
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.3160 | Val Loss: 0.1753 | Val AUC: 0.9998
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold1.pth
Epoch 02/10 | Train Loss: 0.1239 | Val Loss: 0.1825 | Val AUC: 0.9997
Epoch 03/10 | Train Loss: 0.0859 | Val Loss: 0.2374 | Val AUC: 0.9995
Epoch 04/10 | Train Loss: 0.0927 | Val Loss: 0.3527 | Val AUC: 0.9989
Epoch 05/10 | Train Loss: 0.1064 | Val Loss: 0.3932 | Val AUC: 0.9986
Epoch 06/10 | Train Loss: 0.0933 | Val Loss: 0.4439 | Val AUC: 0.9975
Epoch 07/10 | Train Loss: 0.0894 | Val Loss: 0.4582 | Val AUC: 0.9980
Epoch 08/10 | Train Loss: 0.0839 | Val Loss: 0.5026 | Val AUC: 0.9963
Epoch 09/10 | Train Loss: 0.0869 | Val Loss: 0.5270 | Val AUC: 0.9964
Epoch 10/10 | Train Loss: 0.0811 | Val Loss: 0.5437 | Val AUC: 0.9967


100%|██████████| 7110/7110 [00:01<00:00, 3739.13it/s]


TRAINING FOLD 2
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.2504 | Val Loss: 0.1100 | Val AUC: 0.9999
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold2.pth
Epoch 02/10 | Train Loss: 0.0978 | Val Loss: 0.1396 | Val AUC: 0.9999
Epoch 03/10 | Train Loss: 0.0745 | Val Loss: 0.1873 | Val AUC: 0.9998
Epoch 04/10 | Train Loss: 0.0889 | Val Loss: 0.2630 | Val AUC: 0.9994
Epoch 05/10 | Train Loss: 0.0963 | Val Loss: 0.2998 | Val AUC: 0.9989
Epoch 06/10 | Train Loss: 0.0862 | Val Loss: 0.3432 | Val AUC: 0.9984
Epoch 07/10 | Train Loss: 0.0803 | Val Loss: 0.3743 | Val AUC: 0.9981
Epoch 08/10 | Train Loss: 0.0782 | Val Loss: 0.4044 | Val AUC: 0.9980
Epoch 09/10 | Train Loss: 0.0802 | Val Loss: 0.4150 | Val AUC: 0.9978
Epoch 10/10 | Train Loss: 0.0803 | Val Loss: 0.4491 | Val AUC: 0.9965


100%|██████████| 7110/7110 [00:01<00:00, 4490.17it/s]


TRAINING FOLD 3
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.2130 | Val Loss: 0.0960 | Val AUC: 0.9999
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold3.pth
Epoch 02/10 | Train Loss: 0.0903 | Val Loss: 0.1165 | Val AUC: 0.9996
Epoch 03/10 | Train Loss: 0.0584 | Val Loss: 0.1438 | Val AUC: 0.9998
Epoch 04/10 | Train Loss: 0.0672 | Val Loss: 0.2222 | Val AUC: 0.9997
Epoch 05/10 | Train Loss: 0.0901 | Val Loss: 0.2739 | Val AUC: 0.9994
Epoch 06/10 | Train Loss: 0.0851 | Val Loss: 0.3102 | Val AUC: 0.9992
Epoch 07/10 | Train Loss: 0.0748 | Val Loss: 0.3317 | Val AUC: 0.9991
Epoch 08/10 | Train Loss: 0.0652 | Val Loss: 0.3495 | Val AUC: 0.9987
Epoch 09/10 | Train Loss: 0.0664 | Val Loss: 0.4000 | Val AUC: 0.9984
Epoch 10/10 | Train Loss: 0.0736 | Val Loss: 0.4251 | Val AUC: 0.9982


100%|██████████| 7109/7109 [00:01<00:00, 4554.97it/s]


TRAINING FOLD 4
Training on 28440 samples, validating on 7109 samples
Epoch 01/10 | Train Loss: 0.1984 | Val Loss: 0.0782 | Val AUC: 0.9999
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold4.pth
Epoch 02/10 | Train Loss: 0.0763 | Val Loss: 0.0962 | Val AUC: 0.9998
Epoch 03/10 | Train Loss: 0.0607 | Val Loss: 0.1307 | Val AUC: 0.9999
Epoch 04/10 | Train Loss: 0.0685 | Val Loss: 0.1879 | Val AUC: 0.9997
Epoch 05/10 | Train Loss: 0.0754 | Val Loss: 0.2214 | Val AUC: 0.9995
Epoch 06/10 | Train Loss: 0.0737 | Val Loss: 0.2603 | Val AUC: 0.9993
Epoch 07/10 | Train Loss: 0.0720 | Val Loss: 0.2899 | Val AUC: 0.9991
Epoch 08/10 | Train Loss: 0.0628 | Val Loss: 0.3327 | Val AUC: 0.9985
Epoch 09/10 | Train Loss: 0.0624 | Val Loss: 0.3615 | Val AUC: 0.9982
Epoch 10/10 | Train Loss: 0.0716 | Val Loss: 0.3951 | Val AUC: 0.9977
CV score: 0.9979 (+/- 0.0039)
